# 云南白药 000538.SZ — MA5xMA15 双均线策略回测

**流程:** 加载前复权数据 → 计算均线与信号 → 模拟交易 → 七大指标
**复权原因:** 数据范围内有除权除息事件，直接使用未复权数据会导致价格断层和指标失真。

## 1. 加载前复权数据

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = pd.read_csv('000538_daily_fq.csv')
df["trade_date"] = pd.to_datetime(df["trade_date"], format="%Y%m%d")
df = df.sort_values("trade_date").reset_index(drop=True)
close = df["close"].values; dates = df["trade_date"]
print(f"{len(df)} 交易日, {dates.iloc[0].strftime("%Y-%m-%d")} ~ {dates.iloc[-1].strftime("%Y-%m-%d")}")
df.head()

241 交易日, 2025-06-30 ~ 2026-06-26


,ts_code,trade_date,open,high,low,close,pre_close,change,pct_chg,vol,amount
0,000538.SZ,2025-06-30,53.23,53.31,52.96,53.22,53.31,-0.09,-0.1688,57964.24,322691.502
1,000538.SZ,2025-07-01,53.22,53.36,53.06,53.20,53.22,-0.02,-0.0376,42804.17,238615.980
2,000538.SZ,2025-07-02,53.20,53.38,52.95,53.02,53.20,-0.18,-0.3383,54659.90,303999.342
3,000538.SZ,2025-07-03,53.07,53.26,52.76,53.25,53.02,0.23,0.4338,49972.88,278179.474
4,000538.SZ,2025-07-04,53.23,53.34,52.85,53.22,53.25,-0.03,-0.0563,55238.20,307573.917


## 2. MA5xMA15 与交易信号

In [2]:
FAST, SLOW = 5, 15
df["ma_fast"] = df["close"].rolling(FAST).mean()
df["ma_slow"] = df["close"].rolling(SLOW).mean()
df["signal"] = 0
golden = (df["ma_fast"].shift(1) <= df["ma_slow"].shift(1)) & (df["ma_fast"] > df["ma_slow"])
death  = (df["ma_fast"].shift(1) >= df["ma_slow"].shift(1)) & (df["ma_fast"] < df["ma_slow"])
df.loc[golden, "signal"] = 1; df.loc[death, "signal"] = -1
in_pos = False
for i in range(len(df)):
    if df.loc[i,"signal"]==1: in_pos=True
    elif df.loc[i,"signal"]==-1: in_pos=False
    df.loc[i,"position"] = int(in_pos)
print(f"金叉 {(df["signal"]==1).sum()} 次, 死叉 {(df["signal"]==-1).sum()} 次")

金叉 8 次, 死叉 9 次


## 3. 模拟交易

In [3]:
initial_capital = 100000.0; capital = initial_capital; shares = 0
pvals, dret = [], []
for i in range(len(df)):
    p = df.loc[i,"close"]
    if i >= SLOW:
        if df.loc[i,"signal"]==1 and capital>0: shares=capital/p; capital=0
        elif df.loc[i,"signal"]==-1 and shares>0: capital=shares*p; shares=0
    tv = capital + shares * p; pvals.append(tv)
    dret.append(0 if i==0 else (tv-pvals[-2])/pvals[-2])
df["portfolio_value"] = pvals; df["daily_return"] = dret
print(f"最终: ¥{pvals[-1]:,.2f}")

最终: ¥94,085.11


## 4. 七大指标

In [4]:
td = (dates.iloc[-1]-dates.iloc[0]).days; yrs = td/365.25; rf = 0.015
total_r = (pvals[-1]-initial_capital)/initial_capital
cagr = (pvals[-1]/initial_capital)**(1/yrs)-1
bh_cagr = (close[-1]/close[0])**(1/yrs)-1
alpha = cagr - bh_cagr
peak = pvals[0]; mdd = 0
for v in pvals:
    if v>peak: peak=v
    dd = (peak-v)/peak
    if dd>mdd: mdd=dd
rs = pd.Series(dret).iloc[SLOW:]
wr = (rs>0).sum()/len(rs)
pos=rs[rs>0]; neg=rs[rs<0]
plr = pos.mean()/abs(neg.mean()) if len(neg)>0 else 999
sharp_ret = rs.mean()*252; sharp_std = rs.std()*np.sqrt(252)
sharpe = (sharp_ret-rf)/sharp_std if sharp_std>0 else 0
print(f'总收益: {total_r*100:+6.2f}%')
print(f'CAGR: {cagr*100:+6.2f}%')
print(f'Alpha: {alpha*100:+6.2f}%')
print(f'MDD: {mdd*100:6.2f}%')
print(f'胜率: {wr*100:6.1f}%')
print(f'盈亏比: {plr:6.2f}')
print(f'夏普: {sharpe:6.2f}')

总收益:  -5.91%
CAGR:  -5.98%
Alpha:  +5.83%
MDD:  11.71%
胜率:   16.4%
盈亏比:   0.84
夏普:  -1.05


## 5. 可视化

In [5]:
COLOR_UP, COLOR_DOWN = '#ef5350', '#26a69a'
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.05,
    row_heights=[0.65,0.35], subplot_titles=("股价·双均线·交易信号","资金曲线"))
buy = df[df['signal']==1]; sell = df[df['signal']==-1]
fig.add_trace(go.Candlestick(x=dates, open=df['open'], high=df['high'],
    low=df['low'], close=df['close'], name='K线',
    increasing_line_color=COLOR_UP, decreasing_line_color=COLOR_DOWN), row=1, col=1)
fig.add_trace(go.Scatter(x=dates, y=df['ma_fast'], mode='lines',
    line=dict(color='#2196f3',width=1.5), name='MA5'), row=1, col=1)
fig.add_trace(go.Scatter(x=dates, y=df['ma_slow'], mode='lines',
    line=dict(color='#ff9800',width=1.5), name='MA15'), row=1, col=1)
fig.add_trace(go.Scatter(x=buy['trade_date'], y=buy['low']*0.985,
    mode='markers', marker=dict(symbol='triangle-up',size=12,color=COLOR_UP),
    name='买入(金叉)'), row=1, col=1)
fig.add_trace(go.Scatter(x=sell['trade_date'], y=sell['high']*1.015,
    mode='markers', marker=dict(symbol='triangle-down',size=12,color=COLOR_DOWN),
    name='卖出(死叉)'), row=1, col=1)
fig.add_trace(go.Scatter(x=dates, y=df['portfolio_value'], mode='lines',
    line=dict(color='#7c4dff',width=2), name='策略净值',
    fill='tozeroy', fillcolor='rgba(124,77,255,0.06)'), row=2, col=1)
fig.add_hline(y=initial_capital, line_dash='dot',
    line_color='rgba(0,0,0,0.25)', row=2, col=1, annotation_text='本金')
fig.update_layout(
    title=f"<b>云南白药 MA5xMA15 回测</b><br>"
          f"<sup>总收益 -5.9% | CAGR -6.0%</sup>",
    xaxis_rangeslider_visible=False, height=900, hovermode='x unified',
    template='plotly_white')
fig.update_xaxes(rangeselector=dict(buttons=list([
    dict(count=1,label='1月',step='month',stepmode='backward'),
    dict(count=3,label='3月',step='month',stepmode='backward'),
    dict(step='all',label='全部')])))
fig.update_yaxes(title_text='价格', row=1, col=1)
fig.update_yaxes(title_text='资金曲线', row=2, col=1)
fig.show()

## 6. 回测结论

### 七大指标
| 类别 | 指标 | 数值 |
|------|------|------|
| 收益 | 总收益率 | -5.91% |
| 收益 | 年化收益率 | -5.98% |
| 收益 | 超额收益 | +5.83% |
| 风险 | 最大回撤 | 11.71% |
| 质量 | 胜率 | 16.4% |
| 质量 | 盈亏比 | 0.84 |
| 综合 | 夏普比率 | -1.05 |

### 概要
- 区间: 2025-06-30 ~ 2026-06-26 (1.0年)
- 信号: 金叉 8 次, 死叉 9 次
- 买入持有: -11.7%
- 数据已前复权处理，消除除权除息对价格的机械影响

> * 回测未含交易手续费用。
